<a href="https://colab.research.google.com/github/nilaynishant/LLM_Course/blob/main/LLM_Geospatial_Handson_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 LLM for Geospatial Data Processing
## Hands-On Exercises — NESAC Basic Course on Remote Sensing & GIS

**North-East Space Applications Centre (NESAC), ISRO**

---

### Workshop Overview

This notebook contains **5 progressive hands-on exercises** covering the integration of Large Language Models (LLMs) with geospatial data processing workflows.

| Exercise | Topic | Concepts |
|----------|-------|----------|
| 1 | Natural Language → Geospatial Query | LLM API, structured output, geocoding |
| 2 | Vector Feature Extraction & Summarization | GeoPandas, LLM summarization |
| 3 | RAG for Geospatial Documents | Embeddings, FAISS, document QA |


**LLM Backend**: Groq API (Llama 3.1 8B / 70B)  
**Environment**: Google Colab (T4 GPU recommended for Exercise 4)

---

### ⚙️ Setup — Run This First

In [ ]:
# ============================================================
# SETUP: Install all required dependencies
# ============================================================
!pip install -q groq geopandas shapely folium rasterio numpy matplotlib \
              langchain langchain-groq langchain-community faiss-cpu \
              sentence-transformers requests pyproj gdal geopy

print("✅ Installation complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
✅ Installation complete!


In [ ]:
# ============================================================
# API KEY CONFIGURATION
# ============================================================
import os
from getpass import getpass

# Get a FREE Groq API key at: https://console.groq.com
GROQ_API_KEY = getpass("🔑 Enter your Groq API key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Verify connection
from groq import Groq
client = Groq(api_key=GROQ_API_KEY)
test = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Reply with: API connection verified"}],
    max_tokens=20
)
print(f"✅ {test.choices[0].message.content}")

🔑 Enter your Groq API key: ··········
✅ API connection verified


---

## 🟢 Exercise 1: Natural Language → Geospatial Query Engine

### Objective
Build a system that accepts **plain English questions** about geographic locations and returns structured geospatial data (coordinates, bounding boxes, administrative info).

### Learning Goals
- Use LLM to extract structured data from free-text queries  
- Integrate Nominatim geocoding API  
- Parse and validate LLM JSON responses  
- Display results on an interactive Folium map

### Background
Traditional GIS workflows require users to know exact place names, administrative hierarchies, or coordinate systems. LLMs can act as a **natural language interface**, converting ambiguous queries into precise geospatial parameters.

```
User: "Show me flood-prone districts in Assam near the Brahmaputra"
LLM → Extracts: {region: "Assam", feature: "flood-prone districts", river: "Brahmaputra"}
GIS → Geocodes + renders map
```

In [ ]:
# ============================================================
# EXERCISE 1: Natural Language → Geospatial Query Engine
# ============================================================
import json
import requests
import folium
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# ── Step 1: LLM extracts structured location intent ──────────────────────────

EXTRACTION_SYSTEM_PROMPT = """You are a geospatial query parser.
Extract location information from user queries and return ONLY valid JSON.
JSON schema:
{
  "primary_location": "string (city/district/state/country)",
  "country": "string",
  "location_type": "one of: city, district, state, region, country",
  "nearby_feature": "string or null (river/mountain/coast etc)",
  "search_query": "string (best Nominatim search string)",
  "zoom_level": "integer 6-15 (appropriate map zoom)"
}
Return ONLY the JSON object, no other text."""

def extract_location_intent(user_query: str) -> dict:
    """Use LLM to parse natural language into structured location data."""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ],
        max_tokens=300,
        temperature=0.1  # Low temperature for deterministic structured output
    )
    raw = response.choices[0].message.content.strip()
    # Clean potential markdown fences
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

# ── Step 2: Geocode using Nominatim ──────────────────────────────────────────

def geocode_location(search_query: str) -> dict:
    """Geocode a location using OpenStreetMap Nominatim."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": search_query,
        "format": "json",
        "limit": 5,
        "addressdetails": 1
    }
    headers = {"User-Agent": "NESAC-GIS-Workshop/1.0"}
    resp = requests.get(url, params=params, headers=headers)
    results = resp.json()
    if results:
        top = results[0]
        return {
            "lat": float(top["lat"]),
            "lon": float(top["lon"]),
            "display_name": top["display_name"],
            "type": top.get("type", "unknown"),
            "bbox": top.get("boundingbox", [])
        }
    return None

# ── Step 3: LLM generates contextual description ─────────────────────────────

def generate_geo_description(location_name: str, location_type: str) -> str:
    """Ask LLM for a brief geospatial context about the location."""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{
            "role": "user",
            "content": f"""Give a 3-sentence geographic description of {location_name} ({location_type})
            focusing on: terrain, major rivers/water bodies, and relevance to remote sensing/disaster monitoring.
            Be concise and factual."""
        }],
        max_tokens=150
    )
    return response.choices[0].message.content.strip()

# ── Step 4: Build interactive map ────────────────────────────────────────────

def nl_to_geo_map(user_query: str):
    """End-to-end: natural language query → interactive map."""
    print(f"🔍 Query: '{user_query}'")
    print("⏳ Extracting location intent...")

    intent = extract_location_intent(user_query)
    print(f"📋 Extracted intent:\n{json.dumps(intent, indent=2)}")

    geo = geocode_location(intent["search_query"])
    if not geo:
        print("❌ Geocoding failed — location not found")
        return

    print(f"📍 Geocoded: {geo['display_name'][:80]}...")

    description = generate_geo_description(
        intent["primary_location"], intent["location_type"]
    )

    # Create Folium map
    m = folium.Map(
        location=[geo["lat"], geo["lon"]],
        zoom_start=intent["zoom_level"],
        tiles="CartoDB positron"
    )

    # Add marker with popup
    popup_html = f"""
    <div style='width:300px; font-family:Arial'>
      <h4 style='color:#1a5276'>{intent['primary_location']}</h4>
      <b>Type:</b> {intent['location_type']}<br>
      <b>Coords:</b> {geo['lat']:.4f}°N, {geo['lon']:.4f}°E<br>
      <hr>
      <small>{description}</small>
    </div>
    """
    folium.Marker(
        [geo["lat"], geo["lon"]],
        popup=folium.Popup(popup_html, max_width=320),
        icon=folium.Icon(color="red", icon="map-marker")
    ).add_to(m)

    # Add bounding box if available
    if geo["bbox"] and len(geo["bbox"]) == 4:
        south, north, west, east = [float(x) for x in geo["bbox"]]
        folium.Rectangle(
            bounds=[[south, west], [north, east]],
            color="#e74c3c", weight=2, fill=True, fill_opacity=0.05,
            tooltip=f"Bounding Box: {intent['primary_location']}"
        ).add_to(m)

    return m

# ── Run Exercise 1 ────────────────────────────────────────────────────────────
queries = [
    "Show me the capital of Manipur state in Northeast India",
    "I want to study flood dynamics near the Brahmaputra in Assam",
    "Find Kaziranga National Park for wildlife habitat mapping"
]

# Try each query — change the index to switch queries
map_result = nl_to_geo_map(queries[0])
map_result

🔍 Query: 'Show me the capital of Manipur state in Northeast India'
⏳ Extracting location intent...
📋 Extracted intent:
{
  "primary_location": "Imphal",
  "country": "India",
  "location_type": "city",
  "nearby_feature": null,
  "search_query": "Imphal, India",
  "zoom_level": 10
}
📍 Geocoded: Imphal, Lamphelpat, Imphal West, Manipur, 795001, India...


In [ ]:
# ── Exercise 1 Challenge ─────────────────────────────────────────────────────
# Try your own natural language query!
# Examples:
#   "Districts with landslide risk in Meghalaya"
#   "Loktak Lake in Manipur for wetland study"
#   "Indo-Bangladesh border region near Tripura"

your_query = "Type your query here"
result_map = nl_to_geo_map(your_query)
result_map

---

## Exercise 2: Vector Feature Extraction & LLM Summarization

### Objective
Build a pipeline that loads vector geospatial data, performs spatial analysis, and uses an LLM to generate an **intelligent narrative summary** of the results.

### Learning Goals
- Work with GeoDataFrames for spatial analysis
- Create and query synthetic administrative/infrastructure data
- Feed spatial analysis results to LLM for narrative generation
- Build a simple geo-chatbot for vector data Q&A

### Background
Planners and emergency managers often need to quickly understand spatial patterns — e.g., "Which villages are within 5km of the flood zone?" LLMs can convert tabular spatial outputs into clear, actionable narratives.

In [ ]:
# ============================================================
# EXERCISE 3: Vector Feature Extraction & LLM Summarization
# ============================================================
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, MultiPolygon
import folium
from groq import Groq
import json
import numpy as np

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# ── Step 1: Create synthetic Northeast India GeoDataFrame ────────────────────
# Centered around Manipur/Assam region

# Sample districts (approximate centroids)
districts_data = {
    "district": ["Imphal West", "Imphal East", "Bishnupur", "Thoubal", "Churachandpur",
                  "Senapati", "Ukhrul", "Tamenglong", "Chandel", "Jiribam"],
    "state": ["Manipur"] * 10,
    "population": [514683, 456113, 240363, 421764, 274143, 354972, 183998, 140441, 144182, 43852],
    "area_km2": [519, 709, 496, 514, 4570, 3269, 4544, 4391, 3313, 234],
    "flood_risk": ["High", "High", "Very High", "High", "Low", "Low", "Low", "Low", "Low", "Medium"],
    "forest_cover_pct": [12, 18, 8, 15, 65, 72, 81, 78, 68, 42],
    "hospitals": [8, 5, 3, 4, 2, 2, 1, 1, 1, 1],
    "geometry": [
        Point(93.92, 24.82), Point(93.98, 24.76), Point(93.76, 24.63),
        Point(94.02, 24.58), Point(93.67, 24.35), Point(93.83, 25.10),
        Point(94.35, 25.08), Point(93.47, 24.87), Point(94.12, 24.28),
        Point(93.12, 24.89)
    ]
}

gdf = gpd.GeoDataFrame(districts_data, crs="EPSG:4326")

# Add buffer zones to simulate district polygons
gdf_projected = gdf.to_crs("EPSG:32646")  # UTM Zone 46N
gdf_projected["geometry"] = gdf_projected.geometry.buffer(15000)  # 15km radius
gdf = gdf_projected.to_crs("EPSG:4326")

# Add computed fields
gdf["pop_density"] = (gdf["population"] / gdf["area_km2"]).round(1)
gdf["vulnerability_score"] = (
    (gdf["flood_risk"].map({"Very High": 4, "High": 3, "Medium": 2, "Low": 1})) * 0.5 +
    (gdf["pop_density"] / gdf["pop_density"].max()) * 3 +
    (1 - gdf["hospitals"] / gdf["hospitals"].max()) * 2
).round(2)

print(f"✅ GeoDataFrame created: {len(gdf)} districts")
print(gdf[["district", "population", "flood_risk", "forest_cover_pct", "vulnerability_score"]].to_string())

✅ GeoDataFrame created: 10 districts
        district  population flood_risk  forest_cover_pct  vulnerability_score
0    Imphal West      514683       High                12                 4.50
1    Imphal East      456113       High                18                 4.20
2      Bishnupur      240363  Very High                 8                 4.72
3        Thoubal      421764       High                15                 4.98
4  Churachandpur      274143        Low                65                 2.18
5       Senapati      354972        Low                72                 2.33
6         Ukhrul      183998        Low                81                 2.37
7     Tamenglong      140441        Low                78                 2.35
8        Chandel      144182        Low                68                 2.38
9        Jiribam       43852     Medium                42                 3.32


In [ ]:
# ── Step 2: Spatial Analysis Functions ───────────────────────────────────────

def analyze_flood_risk_zones(gdf):
    """Perform spatial analysis on flood-risk districts."""
    high_risk = gdf[gdf["flood_risk"].isin(["High", "Very High"])]

    return {
        "total_districts": len(gdf),
        "high_risk_districts": len(high_risk),
        "high_risk_names": high_risk["district"].tolist(),
        "population_at_risk": int(high_risk["population"].sum()),
        "pct_population_at_risk": round(high_risk["population"].sum() / gdf["population"].sum() * 100, 1),
        "avg_forest_cover_high_risk": round(high_risk["forest_cover_pct"].mean(), 1),
        "avg_forest_cover_low_risk": round(gdf[gdf["flood_risk"]=="Low"]["forest_cover_pct"].mean(), 1),
        "top_vulnerable": gdf.nlargest(3, "vulnerability_score")[["district", "vulnerability_score", "flood_risk"]].to_dict("records"),
        "hospital_deficit_high_risk": round(high_risk["population"].sum() / high_risk["hospitals"].sum()),
        "total_area_at_risk_km2": int(high_risk["area_km2"].sum())
    }

analysis_results = analyze_flood_risk_zones(gdf)
print("📊 Spatial Analysis Results:")
print(json.dumps(analysis_results, indent=2))

# ── Step 3: LLM Narrative Generation ─────────────────────────────────────────

NARRATIVE_SYSTEM = """You are a GIS analyst at NESAC-ISRO specializing in disaster risk assessment
for Northeast India. Generate clear, actionable reports from spatial analysis data.
Write for policy makers and disaster management officials. Use specific numbers from the data."""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": NARRATIVE_SYSTEM},
        {"role": "user", "content": f"""Generate a flood risk assessment report for Manipur districts based on this GIS analysis:

{json.dumps(analysis_results, indent=2)}

Include: (1) Executive Summary, (2) Priority Districts for intervention,
(3) Healthcare infrastructure gaps, (4) Role of forest cover as natural buffer,
(5) Three specific satellite-based monitoring recommendations."""}
    ],
    max_tokens=600
)

narrative = response.choices[0].message.content
print("\n" + "="*60)
print("📋 LLM-GENERATED RISK ASSESSMENT REPORT")
print("="*60)
print(narrative)

📊 Spatial Analysis Results:
{
  "total_districts": 10,
  "high_risk_districts": 4,
  "high_risk_names": [
    "Imphal West",
    "Imphal East",
    "Bishnupur",
    "Thoubal"
  ],
  "population_at_risk": 1632923,
  "pct_population_at_risk": 58.9,
  "avg_forest_cover_high_risk": 13.2,
  "avg_forest_cover_low_risk": 72.8,
  "top_vulnerable": [
    {
      "district": "Thoubal",
      "vulnerability_score": 4.98,
      "flood_risk": "High"
    },
    {
      "district": "Bishnupur",
      "vulnerability_score": 4.72,
      "flood_risk": "Very High"
    },
    {
      "district": "Imphal West",
      "vulnerability_score": 4.5,
      "flood_risk": "High"
    }
  ],
  "hospital_deficit_high_risk": 81646,
  "total_area_at_risk_km2": 2238
}

📋 LLM-GENERATED RISK ASSESSMENT REPORT
**Flood Risk Assessment Report for Manipur Districts**

**Executive Summary**

This report presents a comprehensive flood risk assessment for the districts of Manipur, Northeast India. The analysis reveals that 40% o

In [ ]:
# ── Step 4: Interactive Map Visualization ─────────────────────────────────────

# Color scheme for flood risk
risk_colors = {"Very High": "#c0392b", "High": "#e67e22", "Medium": "#f1c40f", "Low": "#27ae60"}

m = folium.Map(location=[24.7, 93.9], zoom_start=8, tiles="CartoDB positron")

for _, row in gdf.iterrows():
    color = risk_colors.get(row["flood_risk"], "#95a5a6")

    popup_html = f"""
    <div style='font-family:Arial; width:220px'>
      <b style='color:#2c3e50; font-size:14px'>{row['district']}</b><br>
      <hr style='margin:4px 0'>
      🏘️ Population: {row['population']:,}<br>
      ⚠️ Flood Risk: <b style='color:{color}'>{row['flood_risk']}</b><br>
      🌲 Forest Cover: {row['forest_cover_pct']}%<br>
      🏥 Hospitals: {row['hospitals']}<br>
      📊 Vulnerability Score: <b>{row['vulnerability_score']}</b>
    </div>
    """

    folium.GeoJson(
        row.geometry,
        style_function=lambda x, c=color: {
            "fillColor": c, "color": "#2c3e50",
            "weight": 1.5, "fillOpacity": 0.5
        },
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{row['district']} ({row['flood_risk']} Risk)"
    ).add_to(m)

# Legend
legend_html = """
<div style='position:fixed; bottom:30px; left:30px; z-index:1000;
            background:white; padding:12px; border-radius:8px;
            border:1px solid #ccc; font-family:Arial; font-size:12px'>
  <b>Flood Risk Level</b><br>
  <span style='color:#c0392b'>■</span> Very High<br>
  <span style='color:#e67e22'>■</span> High<br>
  <span style='color:#f1c40f'>■</span> Medium<br>
  <span style='color:#27ae60'>■</span> Low
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
m

In [ ]:
# ── Exercise 3 Challenge: Geo-Chatbot ────────────────────────────────────────
# Build a simple question-answering chatbot about the district data

# Serialize GDF to text summary for LLM context
context_table = gdf[["district", "population", "area_km2", "flood_risk",
                       "forest_cover_pct", "hospitals", "vulnerability_score"]].to_string(index=False)

GEO_CHATBOT_SYSTEM = f"""You are a GIS data analyst. Answer questions using ONLY the
following district data from Manipur, India. Be precise and cite numbers.

DATA:
{context_table}
"""

def geo_chatbot(question: str) -> str:
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": GEO_CHATBOT_SYSTEM},
            {"role": "user", "content": question}
        ],
        max_tokens=200
    )
    return resp.choices[0].message.content

# Test questions
questions = [
    "Which district has the highest population density and what is its flood risk?",
    "How many hospitals serve the population in Very High and High risk districts?",
    "Is there a correlation between forest cover and flood risk in this dataset?"
]

for q in questions:
    print(f"\n❓ {q}")
    print(f"💬 {geo_chatbot(q)}")
    print("-" * 50)


❓ Which district has the highest population density and what is its flood risk?
💬 To find the district with the highest population density, we need to divide the population by the area in square kilometers.

Population density = Population / Area (in km2)

For Churachandpur:
Population density = 274143 / 4570 = 60.0 people/km2 (approximately)

For other districts, similar calculations reveal:
Imphal West: 514683 / 519 ≈ 990.8 people/km2
Imphal East: 456113 / 709 ≈ 644.1 people/km2
Bishnupur: 240363 / 496 ≈ 484.6 people/km2
Thoubal: 421764 / 514 ≈ 820.4 people/km2
Tamenglong: 140441 / 4391 ≈ 32.0 people/km2
Ukhrul: 183998 / 4544 ≈ 40.6 people/km2
Senapati: 354972 / 
--------------------------------------------------

❓ How many hospitals serve the population in Very High and High risk districts?
💬 To find the number of hospitals in Very High and High risk districts, we need to identify the districts with high flood risk and calculate the total number of hospitals in those areas.

The d

---

## Exercise 3: RAG for Geospatial Documents

### Objective
Build a **Retrieval Augmented Generation (RAG)** pipeline that can answer questions about remote sensing and GIS documents using semantic search + LLM generation.

### Learning Goals
- Understand document chunking and embedding
- Build a FAISS vector store for similarity search
- Implement the RAG retrieval + generation pipeline
- Evaluate answer quality and trace sources

### Background
Remote sensing organizations have vast technical documentation (manuals, sensor specs, classification guides). RAG enables a chatbot to answer domain-specific questions accurately from these documents without retraining the LLM.

```
Documents → Chunks → Embeddings → Vector Store
Query → Embed → Retrieve Top-K → LLM → Answer with sources
```

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
!pip install langchain-groq

In [ ]:
!pip install langchain-core

In [ ]:
# ============================================================
# EXERCISE 3: RAG for Geospatial Documents
# ============================================================
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate

# ── Step 1: Knowledge Base (Geospatial domain documents) ─────────────────────
# In real usage: load PDFs, technical manuals, field reports

GEO_KNOWLEDGE_BASE = [
    {
        "source": "Sentinel-2 User Guide",
        "content": """Sentinel-2 is a multispectral imaging mission with 13 spectral bands ranging from
        visible and near-infrared to shortwave infrared. It offers spatial resolutions of 10m (bands 2,3,4,8),
        20m (bands 5,6,7,8A,11,12), and 60m (bands 1,9,10). The revisit time is 5 days at the equator with
        two satellites. Sentinel-2A was launched in June 2015, Sentinel-2B in March 2017. The swath width
        is 290km. Level-2A products provide Bottom of Atmosphere (BOA) reflectance after atmospheric correction.
        Band 4 (Red, 665nm) and Band 8 (NIR, 842nm) are commonly used for NDVI computation."""
    },
    {
        "source": "NDVI Interpretation Guide",
        "content": """NDVI (Normalized Difference Vegetation Index) is calculated as (NIR - Red) / (NIR + Red).
        Values range from -1 to +1. Dense healthy vegetation typically shows values between 0.6 and 0.9.
        Sparse vegetation ranges from 0.2 to 0.5. Bare soil shows values around 0.1 to 0.2.
        Water bodies typically show negative NDVI values (-0.1 to -0.3). Snow and clouds can show
        values near zero or slightly positive. In tropical regions like Northeast India, monsoon season
        (June-September) shows peak NDVI values due to lush vegetation. NDVI time series analysis
        can detect crop growth cycles, drought stress, and deforestation events."""
    },
    {
        "source": "Flood Mapping with SAR - NESAC Technical Note",
        "content": """Synthetic Aperture Radar (SAR) is highly effective for flood mapping because it can
        penetrate cloud cover, making it ideal for monsoon flood events in Northeast India. Sentinel-1
        C-band SAR provides 10m resolution in Interferometric Wide (IW) mode with a 250km swath.
        Flood water appears as dark areas in SAR imagery due to specular reflection (smooth water surface
        returns minimal backscatter to the sensor). The change detection approach compares pre-flood and
        post-flood SAR images. VV polarization is better for open water detection; VH polarization helps
        detect flooded vegetation. Google Earth Engine provides Sentinel-1 time series for Brahmaputra
        basin flood monitoring. NESAC has developed automated flood extent mapping algorithms
        specifically calibrated for the floodplains of Assam and Manipur."""
    },
    {
        "source": "Land Use Land Cover Classification Guide",
        "content": """Land Use Land Cover (LULC) classification using satellite imagery involves supervised
        and unsupervised methods. Random Forest classification using Google Earth Engine is widely adopted
        for regional LULC mapping. For Northeast India, key classes include: Dense Forest, Open Forest,
        Scrubland, Grassland, Agricultural cropland, Water bodies, Built-up areas, and Barren land.
        The Overall Accuracy should exceed 85% with Kappa coefficient above 0.8 for publication standards.
        Training samples should be collected through field surveys or high-resolution imagery interpretation.
        Temporal compositing (using median or percentile composites) reduces cloud contamination in
        multi-temporal analysis. ISRO's BHUVAN portal provides national LULC datasets at 1:50,000 scale."""
    },
    {
        "source": "Remote Sensing for Disaster Management - SDMRI Manual",
        "content": """Remote sensing plays a critical role in the complete disaster management cycle:
        preparedness, response, and recovery. Pre-disaster: Hazard zonation maps using terrain analysis
        (DEM-based slope, aspect, drainage), land subsidence monitoring using InSAR, crop vulnerability
        mapping. During disaster: Near-real-time flood/fire extent mapping using SAR and thermal sensors,
        damage assessment from very high resolution imagery (Resourcesat-2, Cartosat-3). Post-disaster:
        Recovery monitoring using NDVI time series to track vegetation recovery, change detection for
        infrastructure reconstruction. ISRO's NRSC provides Disaster Management Support (DMS) program
        with dedicated teams for rapid image processing. ResourceSat-2A LISS-III at 23.5m resolution
        and LISS-IV at 5.8m resolution are operational Indian satellites useful for disaster monitoring."""
    },
    {
        "source": "GIS for Urban Planning",
        "content": """Geographic Information Systems support urban planning through spatial analysis of
        population density, land use zoning, transportation networks, and infrastructure. Urban growth
        modeling uses cellular automata or agent-based models combined with LULC change detection.
        The Urban Heat Island (UHI) effect can be quantified using Landsat Land Surface Temperature (LST)
        products. NDBI (Normalized Difference Built-up Index) uses SWIR and NIR bands to identify
        impervious surfaces. In Guwahati (Assam), urban expansion has been rapid, with built-up area
        growing from 25 km² in 1990 to over 150 km² by 2020. This has increased surface runoff and
        flood frequency. Site suitability analysis for urban expansion integrates multiple spatial
        criteria: slope (<15°), distance from water bodies (>200m), soil type, and existing land use."""
    }
]

# ── Step 2: Create Document objects and chunk ─────────────────────────────────

documents = [
    Document(page_content=doc["content"], metadata={"source": doc["source"]})
    for doc in GEO_KNOWLEDGE_BASE
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"📄 Documents: {len(documents)} → Chunks: {len(chunks)}")

# ── Step 3: Create embeddings and FAISS vector store ──────────────────────────

print("⏳ Creating embeddings (this may take 1-2 minutes on first run)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ Vector store created!")

# ── Step 4: Build RAG Chain ───────────────────────────────────────────────────

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.1
)

RAG_PROMPT = PromptTemplate(
    template="""You are a remote sensing and GIS expert at NESAC-ISRO.
Answer the question using ONLY the provided context. If the answer is not in the context,
say "This information is not in my current knowledge base."

Context:
{context}

Question: {question}

Answer (be specific, cite relevant technical details from the context):""",
    input_variables=["context", "question"]
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": RAG_PROMPT}
)

# ── Step 5: Test the RAG system ───────────────────────────────────────────────

test_questions = [
    "What is the spatial resolution of Sentinel-2 for Band 8 and how is it used for vegetation analysis?",
    "How can SAR imagery be used for flood mapping in Assam during monsoon season?",
    "What accuracy metrics are required for LULC classification in published research?"
]

for question in test_questions:
    result = rag_chain.invoke({"query": question})
    sources = list(set([doc.metadata["source"] for doc in result["source_documents"]]))
    print(f"\n❓ {question}")
    print(f"💬 {result['result']}")
    print(f"📚 Sources: {', '.join(sources)}")
    print("-" * 70)

📄 Documents: 6 → Chunks: 22
⏳ Creating embeddings (this may take 1-2 minutes on first run)...


/tmp/ipykernel_15722/3442210282.py:98: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store created!

❓ What is the spatial resolution of Sentinel-2 for Band 8 and how is it used for vegetation analysis?
💬 According to the context, the spatial resolution of Sentinel-2 for Band 8 is 20m. 

This information is not in my current knowledge base regarding how Band 8 is used for vegetation analysis.
📚 Sources: Sentinel-2 User Guide, Remote Sensing for Disaster Management - SDMRI Manual
----------------------------------------------------------------------

❓ How can SAR imagery be used for flood mapping in Assam during monsoon season?
💬 SAR imagery can be used for flood mapping in Assam during monsoon season due to its ability to penetrate cloud cover. Sentinel-1 C-band SAR provides 10m resolution in Interferometric Wide (IW) mode with a 250km swath, making it suitable for large-scale flood mapping in the region.

Flood water appears as dark areas in SAR imagery due to specular reflection, which can be utilized for change detection approach. NESAC has developed autom

In [ ]:
# ── Exercise 3 Challenge: Add your own document ───────────────────────────────
# Add a new technical document to the knowledge base and query it

YOUR_DOCUMENT = {
    "source": "Your Document Title",
    "content": """Paste your technical text here. For example, content from a
    sensor manual, research paper abstract, or field report about
    Northeast India remote sensing applications."""
}

# Add to existing vector store
new_doc = Document(
    page_content=YOUR_DOCUMENT["content"],
    metadata={"source": YOUR_DOCUMENT["source"]}
)
new_chunks = splitter.split_documents([new_doc])
vectorstore.add_documents(new_chunks)

# Now query about your new document
your_question = "Ask something about the content you just added"
result = rag_chain.invoke({"query": your_question})
print(f"💬 {result['result']}")
print(f"📚 Sources: {[d.metadata['source'] for d in result['source_documents']]}")

💬 Can you explain how the BHUVAN portal's national LULC (Land Use Land Cover) datasets at 1:50,000 scale can be utilized for multi-temporal analysis in Northeast India?
📚 Sources: ['Your Document Title', 'Land Use Land Cover Classification Guide', 'Land Use Land Cover Classification Guide']


---

## Exercise 4: Agentic Geo-Workflow — LLM as Spatial Analyst

### Objective
Build an **agentic LLM system** where the model autonomously selects and executes geospatial tools to answer complex multi-step analytical questions.

### Learning Goals
- Implement function/tool calling with Groq API
- Define geospatial tools (buffer, intersect, classify, statistics)
- Enable LLM to reason about which tools to use and in what sequence
- Handle multi-step spatial analysis through agentic loops

### Background
An **agent** is an LLM that can use tools. For geospatial AI, this means the LLM can call GIS functions (buffer analysis, intersect, classify) autonomously to answer complex questions like *"Find all villages within 10km of flood zones that have fewer than 2 hospitals."*

```
User Query → LLM decides tool sequence → Execute tools → LLM synthesizes → Final answer
```

In [ ]:
# ============================================================
# EXERCISE 5: Agentic Geo-Workflow
# ============================================================
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# ── Step 1: Define Geospatial Tools ───────────────────────────────────────────
# These are real Python functions the agent can call

# Reuse the GeoDataFrame from Exercise 3
# (Run Exercise 3 first, or recreate gdf here)

def get_districts_summary() -> dict:
    """Get a summary of all districts in the dataset."""
    return {
        "count": len(gdf),
        "districts": gdf[["district", "population", "flood_risk",
                           "forest_cover_pct", "hospitals", "area_km2"]].to_dict("records")
    }

def filter_by_flood_risk(risk_levels: list) -> dict:
    """Filter districts by flood risk level. Levels: Very High, High, Medium, Low"""
    filtered = gdf[gdf["flood_risk"].isin(risk_levels)]
    return {
        "matching_districts": filtered["district"].tolist(),
        "count": len(filtered),
        "total_population": int(filtered["population"].sum()),
        "avg_hospital_count": round(filtered["hospitals"].mean(), 1)
    }

def compute_spatial_buffer_analysis(district_name: str, buffer_km: float) -> dict:
    """Find which other districts overlap with a buffer around a given district."""
    target = gdf[gdf["district"] == district_name]
    if target.empty:
        return {"error": f"District '{district_name}' not found"}

    target_proj = target.to_crs("EPSG:32646")
    buffer_geom = target_proj.geometry.values[0].buffer(buffer_km * 1000)

    gdf_proj = gdf.to_crs("EPSG:32646")
    nearby = gdf_proj[gdf_proj.geometry.intersects(buffer_geom)]
    nearby = nearby[nearby["district"] != district_name]

    return {
        "center_district": district_name,
        "buffer_km": buffer_km,
        "nearby_districts": nearby["district"].tolist(),
        "nearby_count": len(nearby),
        "nearby_total_population": int(nearby["population"].sum())
    }

def rank_districts_by_vulnerability(top_n: int = 5) -> dict:
    """Rank districts by vulnerability score and return top N."""
    ranked = gdf.nlargest(top_n, "vulnerability_score")[
        ["district", "vulnerability_score", "flood_risk", "population", "hospitals"]
    ]
    return {
        "top_vulnerable": ranked.to_dict("records"),
        "ranking_criterion": "Combined flood risk, population density, hospital deficit"
    }

def calculate_resource_gap(population_per_hospital_threshold: int = 80000) -> dict:
    """Identify districts with critical healthcare resource gaps."""
    gdf["pop_per_hospital"] = gdf["population"] / gdf["hospitals"]
    gap_districts = gdf[gdf["pop_per_hospital"] > population_per_hospital_threshold]
    return {
        "threshold_pop_per_hospital": population_per_hospital_threshold,
        "districts_with_gap": gap_districts[["district", "population",
                                              "hospitals", "pop_per_hospital"]].round(0).to_dict("records"),
        "count": len(gap_districts)
    }

# ── Step 2: Tool Definitions for LLM ──────────────────────────────────────────

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_districts_summary",
            "description": "Get summary of all districts including population, flood risk, forest cover, and hospitals",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "filter_by_flood_risk",
            "description": "Filter districts by flood risk level",
            "parameters": {
                "type": "object",
                "properties": {
                    "risk_levels": {
                        "type": "array",
                        "items": {"type": "string", "enum": ["Very High", "High", "Medium", "Low"]},
                        "description": "List of flood risk levels to filter by"
                    }
                },
                "required": ["risk_levels"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compute_spatial_buffer_analysis",
            "description": "Find districts within a certain km radius of a named district",
            "parameters": {
                "type": "object",
                "properties": {
                    "district_name": {"type": "string", "description": "Name of center district"},
                    "buffer_km": {"type": "number", "description": "Buffer radius in kilometers"}
                },
                "required": ["district_name", "buffer_km"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rank_districts_by_vulnerability",
            "description": "Rank and return top N most vulnerable districts",
            "parameters": {
                "type": "object",
                "properties": {
                    "top_n": {"type": "integer", "description": "Number of top districts to return", "default": 5}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_resource_gap",
            "description": "Find districts with critical healthcare resource deficits relative to population",
            "parameters": {
                "type": "object",
                "properties": {
                    "population_per_hospital_threshold": {
                        "type": "integer",
                        "description": "Max acceptable population per hospital",
                        "default": 80000
                    }
                },
                "required": []
            }
        }
    }
]

# ── Step 3: Tool Executor ─────────────────────────────────────────────────────

TOOL_REGISTRY = {
    "get_districts_summary": get_districts_summary,
    "filter_by_flood_risk": filter_by_flood_risk,
    "compute_spatial_buffer_analysis": compute_spatial_buffer_analysis,
    "rank_districts_by_vulnerability": rank_districts_by_vulnerability,
    "calculate_resource_gap": calculate_resource_gap
}

def execute_tool(tool_name: str, tool_args: dict) -> str:
    """Execute a tool and return result as JSON string."""
    if tool_name not in TOOL_REGISTRY:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})
    try:
        result = TOOL_REGISTRY[tool_name](**tool_args)
        return json.dumps(result, default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})

# ── Step 4: Agentic Loop ──────────────────────────────────────────────────────

AGENT_SYSTEM = """You are a GIS analyst for disaster preparedness in Manipur, India.
You have access to geospatial analysis tools. Use them to answer questions comprehensively.
Always use tools to get actual data before answering. Synthesize tool results into clear recommendations."""

def run_geo_agent(user_query: str, max_iterations: int = 5) -> str:
    """Run the agentic geo-workflow loop."""
    print(f"\n🤖 AGENT QUERY: {user_query}")
    print("=" * 60)

    messages = [
        {"role": "system", "content": AGENT_SYSTEM},
        {"role": "user", "content": user_query}
    ]

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",  # 70B for better tool use reasoning
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            max_tokens=1000
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        # If no tool calls, agent is done
        if finish == "stop" or not msg.tool_calls:
            print(f"\n📋 FINAL ANSWER:")
            print(msg.content)
            return msg.content

        # Process tool calls
        messages.append({"role": "assistant", "content": msg.content, "tool_calls": msg.tool_calls})

        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"\n🔧 Calling tool: {tool_name}({tool_args})")
            tool_result = execute_tool(tool_name, tool_args)

            result_dict = json.loads(tool_result)
            # Print a brief summary
            if "error" not in result_dict:
                keys = list(result_dict.keys())[:3]
                print(f"   ✅ Result keys: {keys}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

    return "Agent reached maximum iterations"

# ── Run the Agent ─────────────────────────────────────────────────────────────
agent_query = """Which districts should be prioritized for emergency hospital setup?
Consider flood risk, current hospital capacity, and proximity to the most vulnerable district."""

answer = run_geo_agent(agent_query)


🤖 AGENT QUERY: Which districts should be prioritized for emergency hospital setup? 
Consider flood risk, current hospital capacity, and proximity to the most vulnerable district.

🔧 Calling tool: rank_districts_by_vulnerability({'top_n': 5})
   ✅ Result keys: ['top_vulnerable', 'ranking_criterion']

🔧 Calling tool: filter_by_flood_risk({'risk_levels': ['Very High', 'High']})
   ✅ Result keys: ['matching_districts', 'count', 'total_population']

🔧 Calling tool: calculate_resource_gap({'population_per_hospital_threshold': 80000})
   ✅ Result keys: ['threshold_pop_per_hospital', 'districts_with_gap', 'count']

🔧 Calling tool: compute_spatial_buffer_analysis({'buffer_km': 50, 'district_name': 'Most vulnerable district'})

📋 FINAL ANSWER:
Based on the vulnerability scores and flood risk levels, the top 5 most vulnerable districts in Manipur are Thoubal, Bishnupur, Imphal West, Imphal East, and Jiribam. 

Considering the high flood risk and hospital capacity, the districts that should be pr

In [ ]:
# ── Exercise 5 Challenge: Complex Multi-Tool Query ────────────────────────────
# Try these complex queries that require the agent to chain multiple tools:

complex_queries = [
    """Identify a cluster of high-risk districts: find flood-prone areas, check which
    neighbors are within 50km of the most vulnerable district, and assess whether
    those neighbors also have hospital resource gaps.""",

    """Create a prioritized disaster response plan: rank the top 3 vulnerable districts,
    calculate their healthcare gaps, and recommend resource reallocation from low-risk districts.""",

    """If a major flood event occurs in Bishnupur, which districts would be directly
    affected within 40km, what is the total at-risk population, and how many hospitals
    are available in that impact zone?"""
]

# Change index to try different queries
run_geo_agent(complex_queries[0])


🤖 AGENT QUERY: Identify a cluster of high-risk districts: find flood-prone areas, check which 
    neighbors are within 50km of the most vulnerable district, and assess whether 
    those neighbors also have hospital resource gaps.

🔧 Calling tool: rank_districts_by_vulnerability({})
   ✅ Result keys: ['top_vulnerable', 'ranking_criterion']

🔧 Calling tool: filter_by_flood_risk({'risk_levels': ['Very High', 'High']})
   ✅ Result keys: ['matching_districts', 'count', 'total_population']

🔧 Calling tool: compute_spatial_buffer_analysis({'buffer_km': 50, 'district_name': 'most vulnerable district'})

🔧 Calling tool: calculate_resource_gap({})
   ✅ Result keys: ['threshold_pop_per_hospital', 'districts_with_gap', 'count']

📋 FINAL ANSWER:
Based on the results, the most vulnerable district is Thoubal. The buffer analysis identified Imphal West, Imphal East, Bishnupur, and Thoubal as neighboring districts within 50km of Thoubal. 

Among these neighboring districts, Imphal East, Bishnupur, a

'Based on the results, the most vulnerable district is Thoubal. The buffer analysis identified Imphal West, Imphal East, Bishnupur, and Thoubal as neighboring districts within 50km of Thoubal. \n\nAmong these neighboring districts, Imphal East, Bishnupur, and Thoubal have hospital resource gaps, with population per hospital ratios exceeding the threshold of 80,000. \n\nTherefore, these three districts (Imphal East, Bishnupur, and Thoubal) form a cluster of high-risk districts that require urgent attention to address their flood risk and healthcare resource deficits.'

---

## 📝 Summary & Key Takeaways

| Exercise | What You Built | Key LLM Technique |
|----------|---------------|-------------------|
| **1** | NL → Geospatial Query Engine | Structured JSON output, prompt engineering |
| **2** | Raster Metadata Interpreter | Context injection, domain-specific prompting |
| **3** | Vector Analysis + Geo-Chatbot | Data serialization, context window management |
| **4** | RAG for Remote Sensing Docs | Embeddings, semantic retrieval, source attribution |
| **5** | Agentic Geo-Workflow | Tool calling, multi-step reasoning, agentic loops |

### 🔑 LLM + Geospatial Integration Principles

1. **LLMs as Interfaces**: Convert natural language → structured GIS parameters  
2. **LLMs as Interpreters**: Transform numerical results → human-readable insights  
3. **LLMs as Knowledge Bases**: RAG unlocks domain knowledge from technical documents  
4. **LLMs as Agents**: Autonomous multi-step geospatial workflows with tool use  

### 🚀 Next Steps

- **Connect to real data**: Replace synthetic data with Sentinel-2 downloads via `sentinelsat` or Google Earth Engine  
- **Scale RAG**: Add your organization's technical documents, sensor manuals, and field reports  
- **Deploy**: Wrap these exercises into a Streamlit or Gradio web app for non-technical users  
- **Multimodal**: Use vision models (LLaVA, GPT-4V) for direct satellite image interpretation  

---
